# Clase 096 — DBSCAN

Clustering basado en densidad: descubre clusters de forma arbitraria e identifica outliers (`-1`) sin predefinir `k`. Elegimos `eps` con un k-distance plot.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

## 1. DBSCAN sobre `make_moons`

Dos lunas entrelazadas: el caso donde K-Means falla y DBSCAN brilla. La etiqueta `-1` es ruido.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN

np.random.seed(42)
X, y = make_moons(n_samples=1000, noise=0.05, random_state=42)

db = DBSCAN(eps=0.2, min_samples=5).fit(X)
n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
n_ruido = int((db.labels_ == -1).sum())
print(f"clusters encontrados: {n_clusters} | puntos ruido (-1): {n_ruido}")
assert n_clusters == 2, "DBSCAN deberia separar las dos lunas"

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=db.labels_, cmap="coolwarm", s=12)
plt.title("DBSCAN(eps=0.2, min_samples=5) separa las lunas")
plt.tight_layout(); plt.show()

## 2. K-distance plot para elegir `eps`

Distancia al `min_samples`-ésimo vecino, ordenada. El codo marca un `eps` razonable (no a ojo).

In [ ]:
from sklearn.neighbors import NearestNeighbors

min_samples = 5
nn = NearestNeighbors(n_neighbors=min_samples).fit(X)
dist, _ = nn.kneighbors(X)
kdist = np.sort(dist[:, -1])

plt.figure(figsize=(7, 4))
plt.plot(kdist, color="#37a")
plt.axhline(0.2, ls="--", color="#c33", lw=0.8, label="eps ~ 0.2 (codo)")
plt.xlabel("puntos ordenados"); plt.ylabel(f"dist. al {min_samples}-esimo vecino")
plt.title("K-distance plot: el codo sugiere eps")
plt.legend(); plt.tight_layout(); plt.show()
print("codo aproximado en dist =", round(kdist[int(0.95*len(kdist))], 3))

## 3. Sensibilidad a `eps`

`eps` chico = todo ruido; `eps` grande = un solo cluster. Es el botón más sensible.

In [ ]:
print(f"{'eps':>6} {'clusters':>9} {'% ruido':>9}")
for eps in (0.05, 0.1, 0.2, 0.5):
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(X)
    nc = len(set(lab)) - (1 if -1 in lab else 0)
    pr = 100 * (lab == -1).mean()
    print(f"{eps:>6} {nc:>9} {pr:>8.1f}%")
print("\neps chico -> mucho ruido; eps grande -> se fusiona todo.")

## 4. DBSCAN vs K-Means en datos no convexos

K-Means parte las lunas por la mitad; DBSCAN respeta la forma.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

lab_km = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X)
lab_db = DBSCAN(eps=0.2, min_samples=5).fit_predict(X)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(X[:, 0], X[:, 1], c=lab_km, cmap="coolwarm", s=10)
ax[0].set_title(f"K-Means (ARI={adjusted_rand_score(y, lab_km):.2f})")
ax[1].scatter(X[:, 0], X[:, 1], c=lab_db, cmap="coolwarm", s=10)
ax[1].set_title(f"DBSCAN (ARI={adjusted_rand_score(y, lab_db):.2f})")
plt.tight_layout(); plt.show()
assert adjusted_rand_score(y, lab_db) > adjusted_rand_score(y, lab_km)

## 5. Detección de outliers inyectados

Metemos outliers uniformes y verificamos que DBSCAN los recupera vía la etiqueta `-1`.

In [ ]:
rng = np.random.default_rng(42)
outliers = rng.uniform([-2.5, -2.0], [3.5, 2.5], size=(50, 2))
X_out = np.vstack([X, outliers])
es_outlier = np.r_[np.zeros(len(X), bool), np.ones(len(outliers), bool)]

lab = DBSCAN(eps=0.2, min_samples=5).fit_predict(X_out)
detectado = lab == -1
recall = (detectado & es_outlier).sum() / es_outlier.sum()
print(f"outliers inyectados: {es_outlier.sum()} | recuperados: {int((detectado & es_outlier).sum())}")
print(f"recall de outliers: {recall:.2%}")
assert recall >= 0.80, "DBSCAN deberia recuperar >=80% de los outliers inyectados"

plt.figure(figsize=(7, 5))
plt.scatter(X_out[~detectado, 0], X_out[~detectado, 1], c="#37a", s=8, label="cluster")
plt.scatter(X_out[detectado, 0], X_out[detectado, 1], c="#c33", s=25, marker="x", label="outlier (-1)")
plt.legend(); plt.title("DBSCAN detecta outliers sin modelo aparte")
plt.tight_layout(); plt.show()

## Ejercicios

1. Entrená `DBSCAN(eps=0.2, min_samples=5)` sobre `make_moons` y contá cuántos puntos quedaron en `-1`.
2. Hacé el k-distance plot, identificá el codo y usalo como `eps`.
3. Probá `eps ∈ {0.05, 0.1, 0.2, 0.5}` y reportá número de clusters y % de ruido.
4. Compará DBSCAN vs K-Means (`k=2`) sobre las lunas: mostrá que K-Means las parte mal.

## Conclusiones

- DBSCAN encuentra clusters de forma arbitraria sin fijar `k` y detecta outliers como `-1` en la misma pasada.
- Elegí `eps` con el **k-distance plot** (el codo), no a ojo; escalá siempre las features antes.
- `min_samples ≈ 2·dim` es una regla práctica; `eps` mueve mucho más la aguja.
- En alta dimensión DBSCAN degrada (curse of dimensionality); ahí conviene reducir con PCA o usar HDBSCAN.

## ✅ Soluciones de los ejercicios

Cinco ejercicios de DBSCAN: clustering por densidad, elección de `eps` con el k-distance plot, sensibilidad, comparación con K-Means y HDBSCAN (de `sklearn.cluster`) para densidades mixtas. `n_jobs=1`.

**Ejercicio 1 — DBSCAN sobre moons.** `eps=0.2, min_samples=5`; contamos el ruido (etiqueta `-1`).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_blobs
from sklearn.cluster import DBSCAN, KMeans, HDBSCAN
from sklearn.neighbors import NearestNeighbors

X, y = make_moons(n_samples=500, noise=0.06, random_state=42)
db = DBSCAN(eps=0.2, min_samples=5).fit(X)
n_ruido = int((db.labels_ == -1).sum())
n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
print(f'clusters: {n_clusters} | puntos de ruido (-1): {n_ruido}')
plt.figure(figsize=(6, 4))
plt.scatter(X[:, 0], X[:, 1], c=db.labels_, cmap='coolwarm', s=10)
plt.title('DBSCAN separa las dos lunas'); plt.tight_layout(); plt.show()

**Ejercicio 2 — K-distance plot.** Distancia al k-ésimo vecino (`k=min_samples`) ordenada; el codo sugiere `eps`.

In [ ]:
k = 5
nn = NearestNeighbors(n_neighbors=k).fit(X)
dist, _ = nn.kneighbors(X)
kd = np.sort(dist[:, k - 1])
plt.figure(figsize=(6, 4))
plt.plot(kd); plt.axhline(0.2, color='r', ls='--', label='eps~0.2')
plt.xlabel('puntos ordenados'); plt.ylabel(f'dist. al {k}-NN')
plt.title('K-distance plot: el codo marca eps'); plt.legend(); plt.tight_layout(); plt.show()
codo = kd[int(len(kd) * 0.95)]
print(f'eps sugerido por el codo (~percentil 95): {codo:.3f}')

**Ejercicio 3 — Sensibilidad a `eps`.** `eps` chico → todo ruido; grande → un solo cluster.

In [ ]:
for eps in [0.05, 0.1, 0.2, 0.5]:
    m = DBSCAN(eps=eps, min_samples=5).fit(X)
    nc = len(set(m.labels_)) - (1 if -1 in m.labels_ else 0)
    ruido = (m.labels_ == -1).mean()
    print(f'  eps={eps:<4} -> clusters {nc} | ruido {ruido:.1%}')
print('eps controla el radio de vecindad: es EL hiperparametro critico de DBSCAN.')

**Ejercicio 4 — DBSCAN vs K-Means.** K-Means parte las lunas; DBSCAN las separa por densidad.

In [ ]:
lab_km = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X)
lab_db = DBSCAN(eps=0.2, min_samples=5).fit_predict(X)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(X[:, 0], X[:, 1], c=lab_km, cmap='coolwarm', s=10); ax[0].set_title('K-Means')
ax[1].scatter(X[:, 0], X[:, 1], c=lab_db, cmap='coolwarm', s=10); ax[1].set_title('DBSCAN')
plt.tight_layout(); plt.show()
print('DBSCAN sigue la forma de los datos; K-Means impone fronteras rectas.')

**Ejercicio 5 — HDBSCAN y densidades mixtas.** Con blobs de `cluster_std` distinto, un `eps` único de DBSCAN no sirve para ambos; HDBSCAN adapta la densidad.

In [ ]:
Xb, _ = make_blobs(n_samples=800, centers=[[0, 0], [8, 8]],
                   cluster_std=[0.4, 2.0], random_state=42)
db = DBSCAN(eps=0.5, min_samples=5).fit(Xb)
hdb = HDBSCAN(min_cluster_size=20).fit(Xb)
nc_db = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
nc_hdb = len(set(hdb.labels_)) - (1 if -1 in hdb.labels_ else 0)
print(f'DBSCAN (eps unico): {nc_db} clusters | ruido {(db.labels_==-1).mean():.1%}')
print(f'HDBSCAN           : {nc_hdb} clusters | ruido {(hdb.labels_==-1).mean():.1%}')
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(Xb[:, 0], Xb[:, 1], c=db.labels_, cmap='tab10', s=10); ax[0].set_title('DBSCAN')
ax[1].scatter(Xb[:, 0], Xb[:, 1], c=hdb.labels_, cmap='tab10', s=10); ax[1].set_title('HDBSCAN')
plt.tight_layout(); plt.show()
print('HDBSCAN varia la densidad por region: captura ambos blobs sin tunear eps.')